In [2]:
import pandas as pd


In [3]:
df = pd.read_csv("alternative_credit_scoring.csv")

print(df.head())
print(df.shape)
print(df.columns)

   age  monthly_income_inr   area_type  upi_monthly_avg_txn_count  \
0   56               28426       Rural                         57   
1   46               20772  Semi-Urban                        113   
2   32                8450       Rural                         40   
3   60                6712       Urban                          5   
4   25                4367       Urban                         74   

   upi_avg_txn_amount  upi_failed_txn_ratio  \
0             1207.68                 0.024   
1             1194.25                 0.067   
2              373.44                 0.041   
3               85.96                 0.220   
4              127.76                 0.347   

   mobile_recharge_consistency_score  mobile_recharge_lapse_days_max  \
0                               0.98                              75   
1                               0.38                              30   
2                               0.64                              11   
3             

In [4]:
rename_columns = {
    "monthly_income_inr": "monthly_income",
    "upi_monthly_avg_txn_count": "digital_txn_monthly_count",
    "upi_avg_txn_amount": "digital_txn_avg_amount",
    "upi_failed_txn_ratio": "digital_txn_failed_ratio",
    "gig_monthly_earnings": "additional_monthly_income",
    "gig_platform_rating": "additional_income_reliability_score",
    "jandhan_avg_balance": "savings_account_avg_balance",
    "jandhan_zero_balance_months": "zero_balance_months",
    "banking_app_login_frequency_monthly": "mobile_banking_usage_monthly",
    "fintech_apps_count": "finance_apps_count",
    "shg_savings_amount_inr": "group_savings_amount",
    "electricity_bill_avg_amount": "electricity_bill_avg_amount",
    "credit_risk_label": "risk_level"
}

df = df.rename(columns=rename_columns)

print(df.columns)

Index(['age', 'monthly_income', 'area_type', 'digital_txn_monthly_count',
       'digital_txn_avg_amount', 'digital_txn_failed_ratio',
       'mobile_recharge_consistency_score', 'mobile_recharge_lapse_days_max',
       'utility_bills_on_time_pct', 'electricity_bill_avg_amount',
       'cooperative_meeting_attendance_pct', 'group_savings_amount',
       'microfinance_loans_repaid_count', 'microfinance_default_rate',
       'additional_monthly_income', 'additional_income_reliability_score',
       'savings_account_avg_balance', 'zero_balance_months',
       'monthly_data_usage_gb', 'telecom_bill_late_payments',
       'mobile_banking_usage_monthly', 'finance_apps_count',
       'rent_on_time_payment_pct', 'rent_to_income_ratio',
       'social_referral_count', 'insurance_policy_count',
       'alternative_credit_score', 'risk_level', 'loan_approved'],
      dtype='object')


In [5]:
X = df.drop(columns=["risk_level", "loan_approved"])
y = df["risk_level"]

In [6]:
print(X.head())
print(y.value_counts())

   age  monthly_income   area_type  digital_txn_monthly_count  \
0   56           28426       Rural                         57   
1   46           20772  Semi-Urban                        113   
2   32            8450       Rural                         40   
3   60            6712       Urban                          5   
4   25            4367       Urban                         74   

   digital_txn_avg_amount  digital_txn_failed_ratio  \
0                 1207.68                     0.024   
1                 1194.25                     0.067   
2                  373.44                     0.041   
3                   85.96                     0.220   
4                  127.76                     0.347   

   mobile_recharge_consistency_score  mobile_recharge_lapse_days_max  \
0                               0.98                              75   
1                               0.38                              30   
2                               0.64                          

In [7]:
X = pd.get_dummies(X, drop_first=True)

print(X.head())
print(X.shape)

   age  monthly_income  digital_txn_monthly_count  digital_txn_avg_amount  \
0   56           28426                         57                 1207.68   
1   46           20772                        113                 1194.25   
2   32            8450                         40                  373.44   
3   60            6712                          5                   85.96   
4   25            4367                         74                  127.76   

   digital_txn_failed_ratio  mobile_recharge_consistency_score  \
0                     0.024                               0.98   
1                     0.067                               0.38   
2                     0.041                               0.64   
3                     0.220                               0.16   
4                     0.347                               0.98   

   mobile_recharge_lapse_days_max  utility_bills_on_time_pct  \
0                              75                       0.75   
1           

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9925

Classification Report:
               precision    recall  f1-score   support

    High Risk       1.00      0.95      0.97        39
     Low Risk       0.99      1.00      1.00       143
  Medium Risk       0.99      1.00      1.00       206
Very Low Risk       1.00      0.92      0.96        12

     accuracy                           0.99       400
    macro avg       1.00      0.97      0.98       400
 weighted avg       0.99      0.99      0.99       400


Confusion Matrix:
[[ 37   0   2   0]
 [  0 143   0   0]
 [  0   0 206   0]
 [  0   1   0  11]]


In [11]:
sample_customer = X.iloc[[0]]

prediction = model.predict(sample_customer)
probability = model.predict_proba(sample_customer)

print("Predicted Risk Level:", prediction[0])
print("Prediction Probabilities:", probability)

Predicted Risk Level: Low Risk
Prediction Probabilities: [[0.005 0.93  0.065 0.   ]]


In [12]:
import joblib

joblib.dump(model, "risk_prediction_model.pkl")
joblib.dump(X.columns.tolist(), "model_columns.pkl")

['model_columns.pkl']

In [17]:
def decision_recommendation(risk_level, monthly_income, rent_to_income_ratio, requested_loan_amount):
    # 1. Calculate disposable income
    disposable_income = monthly_income * (1 - rent_to_income_ratio)

    # 2. Risk-based loan multiplier
    if risk_level == "Very Low Risk":
        max_loan = monthly_income * 8
        decision = "Approve"
    elif risk_level == "Low Risk":
        max_loan = monthly_income * 6
        decision = "Approve"
    elif risk_level == "Medium Risk":
        max_loan = monthly_income * 3
        decision = "Reduce amount"
    else:
        max_loan = monthly_income * 1.5
        decision = "Reject or request guarantor"

    # 3. Final decision based on requested amount
    if requested_loan_amount <= max_loan and risk_level in ["Very Low Risk", "Low Risk"]:
        final_decision = "Approve"
        recommended_loan = requested_loan_amount
    elif requested_loan_amount <= max_loan and risk_level == "Medium Risk":
        final_decision = "Approve with caution"
        recommended_loan = requested_loan_amount
    elif risk_level == "High Risk":
        final_decision = "Reject or request guarantor"
        recommended_loan = max_loan
    else:
        final_decision = "Reduce amount"
        recommended_loan = max_loan

    # 4. Monthly installment
    suggested_monthly_installment = disposable_income * 0.35

    # 5. Repayment duration
    if suggested_monthly_installment > 0:
        repayment_months = recommended_loan / suggested_monthly_installment
    else:
        repayment_months = 0

    return {
        "predicted_risk_level": risk_level,
        "final_decision": final_decision,
        "requested_loan_amount": float(requested_loan_amount),
        "recommended_loan_amount": float(round(recommended_loan, 2)),
        "suggested_monthly_installment": float(round(suggested_monthly_installment, 2)),
        "estimated_repayment_duration_months": int(round(repayment_months))
    }

In [18]:
customer = df.iloc[0]

requested_loan_amount = 200000

result = decision_recommendation(
    risk_level=prediction[0],
    monthly_income=customer["monthly_income"],
    rent_to_income_ratio=customer["rent_to_income_ratio"],
    requested_loan_amount=requested_loan_amount
)

print(result)

{'predicted_risk_level': 'Low Risk', 'final_decision': 'Reduce amount', 'requested_loan_amount': 200000.0, 'recommended_loan_amount': 170556.0, 'suggested_monthly_installment': 3382.69, 'estimated_repayment_duration_months': 50}
